# 仕訳データ → MoneyForward インポートCSV生成
## ルミナス / 医療法人 恵聖会

Google Drive上の仕訳データを読み込み、MoneyForwardインポート用CSVを法人別に生成します。

### 使い方
1. 「ランタイム」→「すべてのセルを実行」
2. Google Driveへのアクセス許可を承認
3. 出力されたCSVをMoneyForwardにインポート

## ① Google Driveをマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive マウント完了')

## ② フォルダパス設定
Google Driveの仕訳データフォルダのパスを指定してください。  
（マイドライブ直下なら `/content/drive/MyDrive/フォルダ名`）

In [ ]:
import os

# ===== ここを編集 =====
# 恵聖会の仕訳データフォルダ
KEISEIKAI_FOLDER = '/content/drive/MyDrive/恵聖会_経理AI共有フォルダ'
# ルミナスの仕訳データフォルダ
LUMINOUS_FOLDER = '/content/drive/MyDrive/ルミナス_経理'
# 出力先フォルダ（Drive上に保存）
OUTPUT_FOLDER = '/content/drive/MyDrive/MFインポートCSV_出力'
# ====================

# フォルダ確認
for name, path in [('恵聖会', KEISEIKAI_FOLDER), ('ルミナス', LUMINOUS_FOLDER)]:
    if os.path.exists(path):
        files = os.listdir(path)
        print(f'✓ {name}: {path} ({len(files)}ファイル)')
        for f in sorted(files)[:10]:
            print(f'    - {f}')
        if len(files) > 10:
            print(f'    ... 他{len(files)-10}ファイル')
    else:
        print(f'⚠ {name}: フォルダが見つかりません → {path}')
        print(f'   上のパスを修正してください')

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f'\n出力先: {OUTPUT_FOLDER}')

## ③ 勘定科目マッピングルール定義

In [ ]:
import re
import csv
import json
from datetime import datetime
from openpyxl import load_workbook

# ===== キーワード→勘定科目ルール（優先順位順）=====
KEYWORD_RULES = [
    # 人件費
    (r'給与|給料|サラリー|キュウヨ', '給料賃金', '対象外'),
    (r'賞与|ボーナス', '給料賃金', '対象外'),
    (r'社会保険|厚生年金|健保|雇用保険|労働保険', '法定福利費', '対象外'),
    # 通信費
    (r'NTT|電話|テレホン|ﾃﾚﾎﾝ', '通信費', '課仕10%'),
    (r'レイヤード|レイヤ-ド|LAYERED', '通信費', '課仕10%'),
    (r'インターネット|ﾌﾚｯﾂ|フレッツ|プロバイダ', '通信費', '課仕10%'),
    (r'切手|郵便|ゆうパック|ﾕｳﾊﾟｯｸ', '通信費', '課仕10%'),
    # 業務委託
    (r'ALSOK|アルソック|綜合警備', '業務委託料', '課仕10%'),
    (r'清掃|クリーニング', '業務委託料', '課仕10%'),
    # 保険
    (r'保険|ホケン|ケンホケンイキョウカイ', '保険料', '対象外'),
    # 備品・消耗品
    (r'アマゾン|AMAZON|ｱﾏｿﾞﾝ', '備品・消耗品費', '課仕10%'),
    (r'楽天市場|ﾗｸﾃﾝ|RAKUTEN', '備品・消耗品費', '課仕10%'),
    (r'消耗品|備品|文具|ぶんぐ', '備品・消耗品費', '課仕10%'),
    # クレジットカード引落
    (r'クレジット|ｸﾚｼﾞｯﾄ|カード引落|ｶｰﾄﾞﾋｷｵﾄ', '未払金', '対象外'),
    (r'JCB|VISA|ﾏｽﾀｰ|MASTER|ﾗｸﾃﾝｶｰﾄﾞ|楽天カード', '未払金', '対象外'),
    # 地代家賃
    (r'家賃|ヤチン|賃料|チンリョウ', '地代家賃', '非課税'),
    # 水道光熱費
    (r'水道|スイドウ|ｽｲﾄﾞｳ', '水道光熱費', '課仕10%'),
    (r'電気|デンキ|ﾃﾞﾝｷ|東京電力|東電|TEPCO', '水道光熱費', '課仕10%'),
    (r'ガス|ｶﾞｽ|東京ガス|大阪ガス', '水道光熱費', '課仕10%'),
    # 医薬品
    (r'医薬品|イヤクヒン|薬|クスリ', '医薬品費', '課仕10%'),
    # 検査委託
    (r'検査|ケンサ|臨床検査', '検査委託費', '課仕10%'),
    # 広告
    (r'広告|コウコク|HP|ホームページ|WEB|ウェブ', '広告宣伝費', '課仕10%'),
    # 修繕
    (r'修繕|シュウゼン|修理|工事', '修繕費', '課仕10%'),
    # リース
    (r'リース|ﾘｰｽ|LEASE|レンタル', 'リース料', '課仕10%'),
    # 支払報酬
    (r'税理士|ゼイリシ|社労士|シャロウシ|弁護士|司法書士|行政書士', '支払報酬', '課仕10%'),
    # 研修
    (r'研修|ケンシュウ|セミナー|講習', '研修費', '課仕10%'),
    # 接待交際費
    (r'接待|セッタイ|交際|贈答', '接待交際費', '課仕10%'),
    # 旅費交通費
    (r'交通費|コウツウヒ|タクシー|新幹線|飛行機|JR|ＪＲ', '旅費交通費', '課仕10%'),
    # 振込手数料
    (r'振込手数料|ﾌﾘｺﾐﾃｽｳﾘｮｳ|手数料|テスウリョウ', '支払手数料', '課仕10%'),
    # 収入系
    (r'診療報酬|社保|国保|シャホ|コクホ', '売上高', '対象外'),
    (r'窓口|マドグチ|自費|ジヒ', '売上高', '課仕10%'),
    # 振替
    (r'振替|フリカエ|口座間', '普通預金', '対象外'),
    # 税金
    (r'源泉|ゲンセン|所得税|住民税|法人税', '租税公課', '対象外'),
    (r'印紙|消費税|固定資産税', '租税公課', '対象外'),
]

def classify_transaction(description):
    """摘要テキストから勘定科目を自動判定"""
    if not description:
        return '不明', '', '要確認'
    desc = str(description).upper()
    for pattern, account, tax in KEYWORD_RULES:
        if re.search(pattern, desc, re.IGNORECASE):
            return account, tax, '自動'
    return '不明', '', '要確認'

print('✓ 勘定科目ルール定義完了（', len(KEYWORD_RULES), 'ルール）')

## ④ データ読み込み＆処理

In [ ]:
def detect_encoding(filepath):
    for enc in ['utf-8', 'shift-jis', 'cp932', 'euc-jp']:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                f.read(1000)
            return enc
        except (UnicodeDecodeError, UnicodeError):
            continue
    return 'utf-8'

def read_excel(filepath):
    """Excelファイルから仕訳データ読み込み"""
    wb = load_workbook(filepath, data_only=True)
    ws = wb.active
    headers = []
    header_row = 1
    for row in ws.iter_rows(min_row=1, max_row=5, values_only=False):
        cells = [cell.value for cell in row]
        if any(h in str(c or '') for c in cells for h in ['日付','取引日','借方','勘定科目','摘要']):
            headers = [str(c or '').strip() for c in cells]
            header_row = row[0].row
            break
    if not headers:
        first_row = next(ws.iter_rows(min_row=1, max_row=1, values_only=True))
        headers = [str(c or '').strip() for c in first_row]
    entries = []
    for row in ws.iter_rows(min_row=header_row + 1, values_only=True):
        if all(c is None for c in row): continue
        entry = {}
        for i, h in enumerate(headers):
            if i < len(row): entry[h] = row[i]
        entries.append(entry)
    wb.close()
    return headers, entries

def read_csv_file(filepath):
    """CSV/TSVファイルから読み込み"""
    encoding = detect_encoding(filepath)
    with open(filepath, 'r', encoding=encoding) as f:
        first_line = f.readline()
        delimiter = '\t' if '\t' in first_line else ','
    with open(filepath, 'r', encoding=encoding) as f:
        reader = csv.reader(f, delimiter=delimiter)
        headers = [h.strip() for h in next(reader, [])]
        entries = []
        for row in reader:
            if not any(row): continue
            entry = {h: row[i] if i < len(row) else '' for i, h in enumerate(headers)}
            entries.append(entry)
    return headers, entries

def normalize_entry(raw):
    """様々な形式を統一フォーマットに正規化"""
    date_keys = ['日付','取引日','date']
    debit_acc_keys = ['借方勘定科目','借方科目']
    debit_amt_keys = ['借方金額','出金','お引出し','引出金額']
    credit_acc_keys = ['貸方勘定科目','貸方科目']
    credit_amt_keys = ['貸方金額','入金','お預入','預入金額']
    tax_keys = ['税区分','tax_category']
    desc_keys = ['摘要','description','備考','お取引内容','取引内容']

    def find(entry, keys):
        for k in keys:
            if k in entry and entry[k] is not None: return entry[k]
        return None

    def parse_amt(val):
        if val is None or val == '': return 0
        if isinstance(val, (int, float)): return abs(int(val))
        try: return abs(int(float(str(val).replace(',','').replace('¥','').replace('￥','').strip())))
        except: return 0

    date = find(raw, date_keys) or ''
    if isinstance(date, datetime): date = date.strftime('%Y/%m/%d')
    else: date = str(date).strip().replace('-','/')

    debit_acc = str(find(raw, debit_acc_keys) or '').strip()
    credit_acc = str(find(raw, credit_acc_keys) or '').strip()
    debit_amt = parse_amt(find(raw, debit_amt_keys))
    credit_amt = parse_amt(find(raw, credit_amt_keys))
    tax = str(find(raw, tax_keys) or '').strip()
    desc = str(find(raw, desc_keys) or '').strip()

    # 科目がない場合は自動分類
    if not debit_acc and not credit_acc and desc:
        is_deposit = credit_amt > 0 and debit_amt == 0
        account, tax_cat, status = classify_transaction(desc)
        if is_deposit:
            debit_acc = '普通預金'
            credit_acc = account if account != '不明' else '売上高'
        else:
            debit_acc = account
            credit_acc = '普通預金'
        tax = tax_cat
    else:
        status = '既存'

    amount = debit_amt or credit_amt
    if debit_amt == 0: debit_amt = credit_amt
    if credit_amt == 0: credit_amt = debit_amt

    return {
        'date': date, 'debit_account': debit_acc, 'debit_amount': debit_amt,
        'credit_account': credit_acc, 'credit_amount': credit_amt,
        'tax_category': tax, 'description': desc, 'status': status
    }

def process_folder(folder_path):
    """フォルダ内の全ファイルを処理"""
    all_entries = []
    if not os.path.exists(folder_path):
        print(f'⚠ フォルダが見つかりません: {folder_path}')
        return all_entries

    for root, dirs, files in os.walk(folder_path):
        for f in sorted(files):
            if f.startswith('~$') or f.startswith('.'): continue
            ext = os.path.splitext(f)[1].lower()
            filepath = os.path.join(root, f)
            try:
                if ext in ('.xlsx', '.xls'):
                    headers, raw = read_excel(filepath)
                elif ext in ('.csv', '.tsv', '.txt'):
                    headers, raw = read_csv_file(filepath)
                else:
                    continue
                entries = [normalize_entry(r) for r in raw]
                entries = [e for e in entries if e.get('date')]
                print(f'  ✓ {f}: {len(entries)}件')
                all_entries.extend(entries)
            except Exception as e:
                print(f'  ⚠ {f}: エラー - {e}')
    return all_entries

# 恵聖会データ処理
print('=== 恵聖会 ===')
keiseikai_entries = process_folder(KEISEIKAI_FOLDER)
print(f'  合計: {len(keiseikai_entries)}件\n')

# ルミナスデータ処理
print('=== ルミナス ===')
luminous_entries = process_folder(LUMINOUS_FOLDER)
print(f'  合計: {len(luminous_entries)}件')

## ⑤ MoneyForward インポートCSV出力

In [ ]:
def write_mf_csv(entries, output_path, entity_name):
    """MFインポート用CSV出力（Shift-JIS）"""
    if not entries:
        print(f'⚠ {entity_name}: データなし、スキップ')
        return None

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='shift-jis', newline='', errors='replace') as f:
        writer = csv.writer(f)
        writer.writerow([
            '取引日','借方勘定科目','借方補助科目','借方税区分','借方金額',
            '貸方勘定科目','貸方補助科目','貸方税区分','貸方金額','摘要'
        ])
        count = 0
        for e in entries:
            if not e.get('date') or not e.get('debit_account'): continue
            writer.writerow([
                e['date'], e['debit_account'], '', e.get('tax_category',''), e['debit_amount'],
                e['credit_account'], '', '', e['credit_amount'], e.get('description','')
            ])
            count += 1

    # 要確認リストも別ファイルで出力
    review = [e for e in entries if e.get('status') == '要確認']
    if review:
        review_path = output_path.replace('.csv', '_要確認.csv')
        with open(review_path, 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['日付','摘要','金額','現在の借方科目','ステータス'])
            for e in review:
                writer.writerow([e['date'], e['description'], e['debit_amount'], e['debit_account'], e['status']])
        print(f'  ⚠ 要確認 {len(review)}件 → {review_path}')

    print(f'  ✓ {entity_name}: {count}件 → {output_path}')
    return output_path

month = datetime.now().strftime('%Y%m')

print('=== MFインポートCSV出力 ===')
k_path = write_mf_csv(keiseikai_entries, os.path.join(OUTPUT_FOLDER, f'MFインポート_恵聖会_{month}.csv'), '恵聖会')
l_path = write_mf_csv(luminous_entries, os.path.join(OUTPUT_FOLDER, f'MFインポート_ルミナス_{month}.csv'), 'ルミナス')

# サマリー
print(f'\n{"="*50}')
print(f'処理完了！')
print(f'  恵聖会: {len(keiseikai_entries)}件')
print(f'  ルミナス: {len(luminous_entries)}件')
auto = sum(1 for e in keiseikai_entries + luminous_entries if e.get('status') in ('自動','既存'))
review = sum(1 for e in keiseikai_entries + luminous_entries if e.get('status') == '要確認')
total = len(keiseikai_entries) + len(luminous_entries)
if total > 0:
    print(f'  自動判定: {auto}件 ({auto*100//total}%)')
    print(f'  要確認: {review}件')
print(f'\n出力先: {OUTPUT_FOLDER}')
print(f'\n次のステップ:')
print(f'  1. 要確認データを確認・修正')
print(f'  2. MFインポートCSVをMoneyForwardにアップロード')
print(f'     MF → 会計帳簿 → 仕訳帳 → インポート')